# Programação Paralela (CCO085) — CUDA no Google Colab
**IESB 2026/2 — Prof. Rodrigo Gonçalves Pinto**

Este roteiro compila e executa código CUDA em C++ usando a GPU gratuita do Colab.

## Antes de começar

Ative a GPU: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware → GPU T4**.

Se a GPU não estiver disponível no momento (o plano gratuito não garante disponibilidade em horário de pico), use o Kaggle Notebooks como alternativa: o mesmo código funciona lá sem alteração.

In [ ]:
!nvidia-smi

A saída acima deve mostrar uma **Tesla T4** com cerca de 15 GB de memória. A T4 tem *compute capability* **7.5** — esse número aparece de novo daqui a pouco, na flag de compilação.

Vamos conferir o compilador CUDA:

In [ ]:
!nvcc --version

## Método 1 — escrever o `.cu` em arquivo e compilar

É o método mais previsível e o que vamos usar em aula. A célula abaixo grava o arquivo no disco da máquina virtual.

In [ ]:
%%writefile soma_vetores.cu
#include <cstdio>

// Kernel: cada thread soma UM elemento.
__global__ void soma(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int N = 1 << 20;              // ~1 milhao de elementos
    size_t bytes = N * sizeof(float);

    float *ha = (float*)malloc(bytes);
    float *hb = (float*)malloc(bytes);
    float *hc = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { ha[i] = i * 1.0f; hb[i] = i * 2.0f; }

    float *da, *db, *dc;
    cudaMalloc(&da, bytes); cudaMalloc(&db, bytes); cudaMalloc(&dc, bytes);

    cudaMemcpy(da, ha, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(db, hb, bytes, cudaMemcpyHostToDevice);

    int threadsPorBloco = 256;
    int blocos = (N + threadsPorBloco - 1) / threadsPorBloco;

    cudaEvent_t ini, fim;
    cudaEventCreate(&ini); cudaEventCreate(&fim);
    cudaEventRecord(ini);
    soma<<<blocos, threadsPorBloco>>>(da, db, dc, N);
    cudaEventRecord(fim);
    cudaEventSynchronize(fim);

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, ini, fim);

    cudaMemcpy(hc, dc, bytes, cudaMemcpyDeviceToHost);

    bool ok = true;
    for (int i = 0; i < N; i++) if (hc[i] != ha[i] + hb[i]) { ok = false; break; }

    printf("blocos=%d  threads/bloco=%d  total=%d\n", blocos, threadsPorBloco, blocos*threadsPorBloco);
    printf("resultado %s\n", ok ? "CORRETO" : "INCORRETO");
    printf("tempo do kernel: %.4f ms\n", ms);

    cudaFree(da); cudaFree(db); cudaFree(dc);
    free(ha); free(hb); free(hc);
    return 0;
}


A flag `-arch=sm_75` diz ao compilador para gerar código para a arquitetura Turing da T4. **Inclua sempre essa flag**: sem ela, dependendo da combinação de versão do toolkit e do driver que o Colab estiver servindo naquele dia, a compilação passa mas o kernel não executa.

In [ ]:
!nvcc -arch=sm_75 -O2 soma_vetores.cu -o soma_vetores

In [ ]:
!./soma_vetores

## Método 2 — escrever CUDA direto na célula

Mais conveniente para experimentos rápidos. Requer o pacote `nvcc4jupyter`.

In [ ]:
!pip install -q nvcc4jupyter
%load_ext nvcc_plugin

In [ ]:
%%cuda
#include <cstdio>

__global__ void quem_sou_eu() {
    int global = blockIdx.x * blockDim.x + threadIdx.x;
    if (global < 8)
        printf("thread %d do bloco %d -> indice global %d\n", threadIdx.x, blockIdx.x, global);
}

int main() {
    quem_sou_eu<<<2, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}


## Exercício 1 — variação da configuração de execução

Volte ao `soma_vetores.cu` e meça o tempo do kernel para `threadsPorBloco` igual a 32, 64, 128, 256, 512 e 1024. Monte a tabela e responda:

1. Qual configuração foi a mais rápida?
2. Por que 32 tende a ser ruim? (dica: pesquise o que é um *warp*)
3. O tempo de kernel que você mediu inclui a transferência host–device? Se não inclui, o que aconteceria com a comparação contra a versão de CPU se você incluísse?


## Exercício 2 — o custo real da GPU

Meça, separadamente:

- o tempo das chamadas `cudaMemcpy` (host → device e device → host);
- o tempo do kernel.

Compare a soma dos dois com o tempo de uma versão sequencial em CPU do mesmo problema. Para `N = 2^20`, a GPU compensa? E para `N = 2^26`?

Escreva duas frases explicando o resultado. Essa é exatamente a discussão que se espera no relatório do Projeto Acadêmico IESB.

---
**Se a GPU não estiver disponível:** abra este mesmo notebook no Kaggle (Notebooks → New → Accelerator: GPU). O `nvidia-smi` pode mostrar uma P100 em vez de T4; nesse caso troque `-arch=sm_75` por `-arch=sm_60`.